# Extract Skills from Job Descriptions

This notebook demonstrates how to extract KSAO skills from job descriptions using semantic matching.

**Features:**
- Semantic similarity matching (handles variations and synonyms)
- Automatic deduplication
- Section categorization (Skills, Abilities, Knowledge, Technology Skills)
- Batch processing with progress tracking

**Use Case:** Extract skills from 20M+ job descriptions to analyze KSAO requirement changes over time.

## Setup

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from pathlib import Path
from skillner.jd_skill_extractor import JobDescriptionSkillExtractor

# For visualization
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

print("✓ Imports successful")

## Configuration

Edit these paths to match your data:

In [ ]:
# Input data
INPUT_DATA = '../data/jd_sampled.parquet'  # Your sampled job descriptions

# Knowledge base (created from ONET)
KB_PATH = '../.skillner-kb/MERGED_EN.pkl'  # Or ONET_EN.pkl

# Output
OUTPUT_PATH = '../data/jd_extracted_skills.parquet'

# Column names in your data
JD_TEXT_COLUMN = 'job_description'  # Column with job description text
ONET_CODE_COLUMN = 'onet_code'
DATE_COLUMN = 'post_date'

# Extraction parameters
SIMILARITY_THRESHOLD = 0.6  # Lower = more matches, higher = stricter
MAX_WINDOW_SIZE = 5  # Maximum words in a skill phrase

print(f"Configuration:")
print(f"  Input: {INPUT_DATA}")
print(f"  KB: {KB_PATH}")
print(f"  Output: {OUTPUT_PATH}")
print(f"  Similarity threshold: {SIMILARITY_THRESHOLD}")
print(f"  Max window size: {MAX_WINDOW_SIZE}")

## Step 1: Load Data

In [ ]:
# Load your sampled job descriptions
print("Loading data...")
df = pd.read_parquet(INPUT_DATA)

print(f"✓ Loaded {len(df):,} job descriptions")
print(f"\nColumns: {list(df.columns)}")
print(f"\nSample:")
df.head(3)

## Step 2: Initialize Skill Extractor

This loads the knowledge base and semantic model (may take 1-2 minutes first time):

In [ ]:
# Initialize extractor (this may take a minute)
extractor = JobDescriptionSkillExtractor(
    kb_path=KB_PATH,
    model_name='all-MiniLM-L6-v2',
    similarity_threshold=SIMILARITY_THRESHOLD,
    max_window_size=MAX_WINDOW_SIZE
)

print("\n✓ Extractor ready!")

## Step 3: Test on Single Example

Let's test on one job description first:

In [ ]:
# Get first non-null job description
test_jd = df[df[JD_TEXT_COLUMN].notna()][JD_TEXT_COLUMN].iloc[0]

print("Test Job Description:")
print("=" * 70)
print(test_jd[:500] + "..." if len(test_jd) > 500 else test_jd)
print("=" * 70)

# Extract skills
result = extractor.extract_skills(test_jd, return_details=True)

print(f"\n✓ Found {result['num_skills']} unique skills")
print(f"\nSkills by category:")
for section, skills in result['by_section'].items():
    print(f"  [{section}]: {len(skills)} skills")
    for skill in skills[:3]:
        print(f"    - {skill}")
    if len(skills) > 3:
        print(f"    ... and {len(skills) - 3} more")

print(f"\nAll extracted skills:")
for i, skill in enumerate(sorted(result['skills']), 1):
    print(f"{i:2d}. {skill}")

## Step 4: Extract Skills from All Job Descriptions

**Note:** This may take several hours depending on dataset size:
- ~2-5 seconds per job description with semantic matching
- For 30,000 JDs: ~17-42 hours
- For 5,000 JDs: ~3-7 hours

In [ ]:
# Extract skills from all job descriptions
print(f"Processing {len(df):,} job descriptions...")
print("This may take several hours. Progress will be shown below.\n")

# Get list of job descriptions
jd_list = df[JD_TEXT_COLUMN].tolist()

# Extract skills with progress bar
results = extractor.extract_skills_batch(jd_list, show_progress=True)

print(f"\n✓ Extraction complete!")

## Step 5: Combine Results with Original Data

In [ ]:
# Convert results to DataFrame
results_df = pd.DataFrame([
    {
        'skills': r['skills'],
        'num_skills': r['num_skills'],
        'by_section': r['by_section']
    }
    for r in results
])

# Combine with original data
df_combined = pd.concat([df.reset_index(drop=True), results_df], axis=1)

# Create quarter column for time series analysis
if DATE_COLUMN in df_combined.columns:
    df_combined[DATE_COLUMN] = pd.to_datetime(df_combined[DATE_COLUMN])
    df_combined['quarter'] = df_combined[DATE_COLUMN].dt.to_period('Q')

print(f"Combined DataFrame shape: {df_combined.shape}")
print(f"\nSample:")
df_combined[['onet_code', 'quarter', 'num_skills']].head()

## Step 6: Basic Statistics

In [ ]:
# Get statistics
stats = extractor.get_statistics(results)

print("="*70)
print("Extraction Statistics")
print("="*70)

print(f"\nTotal job descriptions: {stats['total_jds']:,}")
print(f"Total unique skills found: {stats['unique_skills_total']:,}")

print(f"\nSkills per job description:")
print(f"  Mean: {stats['skills_per_jd']['mean']:.1f}")
print(f"  Median: {stats['skills_per_jd']['median']:.1f}")
print(f"  Min: {stats['skills_per_jd']['min']:.0f}")
print(f"  Max: {stats['skills_per_jd']['max']:.0f}")
print(f"  Std: {stats['skills_per_jd']['std']:.1f}")

print(f"\nSkills by section:")
for section, section_stats in stats['by_section'].items():
    print(f"  [{section}]")
    print(f"    Mean: {section_stats['mean']:.1f}")
    print(f"    Median: {section_stats['median']:.1f}")

print(f"\nTop 10 most common skills:")
for i, (skill, count) in enumerate(stats['top_10_skills'], 1):
    pct = (count / stats['total_jds']) * 100
    print(f"{i:2d}. {skill:40s} ({count:4d} times, {pct:.1f}%)")

## Step 7: Visualizations

In [ ]:
# Distribution of skills per JD
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df_combined['num_skills'], bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Number of Skills per Job Description')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Skills per JD')
axes[0].axvline(stats['skills_per_jd']['mean'], color='red', linestyle='--', label=f"Mean: {stats['skills_per_jd']['mean']:.1f}")
axes[0].legend()

# Box plot by ONET code (top 10)
top_onet = df_combined[ONET_CODE_COLUMN].value_counts().head(10).index
df_top = df_combined[df_combined[ONET_CODE_COLUMN].isin(top_onet)]
df_top.boxplot(column='num_skills', by=ONET_CODE_COLUMN, ax=axes[1], rot=45)
axes[1].set_xlabel('ONET Code')
axes[1].set_ylabel('Number of Skills')
axes[1].set_title('Skills per JD by ONET Code (Top 10)')
plt.suptitle('')

plt.tight_layout()
plt.show()

In [ ]:
# Skills by category
section_counts = {}
for result in results:
    for section, skills in result['by_section'].items():
        section_counts[section] = section_counts.get(section, 0) + len(skills)

plt.figure(figsize=(10, 6))
sections = list(section_counts.keys())
counts = list(section_counts.values())
plt.bar(sections, counts, edgecolor='black', alpha=0.7)
plt.xlabel('KSAO Section')
plt.ylabel('Total Skills Extracted')
plt.title('Skills Extracted by KSAO Category')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\nSkills by section:")
for section, count in sorted(section_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {section:25s}: {count:6,} skills")

## Step 8: Time Series Analysis (Optional)

If you have date information, analyze skill trends over time:

In [ ]:
if 'quarter' in df_combined.columns:
    # Average skills per quarter
    quarterly_stats = df_combined.groupby('quarter')['num_skills'].agg(['mean', 'median', 'count'])
    
    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    
    # Skills over time
    axes[0].plot(quarterly_stats.index.astype(str), quarterly_stats['mean'], marker='o', label='Mean')
    axes[0].plot(quarterly_stats.index.astype(str), quarterly_stats['median'], marker='s', label='Median')
    axes[0].set_xlabel('Quarter')
    axes[0].set_ylabel('Skills per JD')
    axes[0].set_title('Average Skills per Job Description Over Time')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    # Number of JDs per quarter
    axes[1].bar(quarterly_stats.index.astype(str), quarterly_stats['count'], edgecolor='black', alpha=0.7)
    axes[1].set_xlabel('Quarter')
    axes[1].set_ylabel('Number of Job Descriptions')
    axes[1].set_title('Sample Size per Quarter')
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()
    
    print("\nQuarterly statistics:")
    print(quarterly_stats)
else:
    print("No date information available for time series analysis")

## Step 9: Save Results

In [ ]:
# Save combined results
print(f"Saving results to {OUTPUT_PATH}...")
df_combined.to_parquet(OUTPUT_PATH, index=False)
print(f"✓ Saved {len(df_combined):,} records")

# Also save statistics as JSON
import json
stats_path = OUTPUT_PATH.replace('.parquet', '_stats.json')
with open(stats_path, 'w') as f:
    # Convert non-serializable objects
    stats_serializable = {
        'total_jds': stats['total_jds'],
        'unique_skills_total': stats['unique_skills_total'],
        'skills_per_jd': stats['skills_per_jd'],
        'by_section': stats['by_section'],
        'top_10_skills': [[skill, int(count)] for skill, count in stats['top_10_skills']]
    }
    json.dump(stats_serializable, f, indent=2)
print(f"✓ Saved statistics to {stats_path}")

print("\n" + "="*70)
print("EXTRACTION COMPLETE")
print("="*70)
print(f"Processed: {len(df_combined):,} job descriptions")
print(f"Extracted: {stats['unique_skills_total']:,} unique skills")
print(f"Output: {OUTPUT_PATH}")
print(f"Statistics: {stats_path}")

## Next Steps

Now that you have extracted skills, you can:

1. **Analyze specific occupations**: Filter by `onet_code` to see skill requirements for specific jobs
2. **Track skill trends**: Analyze how skill requirements change over quarters
3. **Compare occupations**: Compare skill profiles across different ONET codes
4. **Identify emerging skills**: Find skills that are increasing in frequency over time
5. **KSAO analysis**: Compare requirements across Skills, Knowledge, Abilities, and Technology categories

See `notebooks/jd_temporal_analysis.ipynb` for more advanced temporal analysis examples.